# Models

In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_recall_curve, f1_score, roc_auc_score
import matplotlib.pyplot as plt
import numpy as np

In [2]:
df = pd.read_csv("/Users/jasonc/Desktop/DSC_291/processed_data.csv")

In [3]:
df["customer_category"] = df["masked_consumer_id"].astype(str).str[:3]

In [4]:
# Group by customer_category and FPF_TARGET, then count occurrences
category_target_counts = (
    df.groupby(["customer_category", "FPF_TARGET"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0.0: "count_0", 1.0: "count_1"})
)
category_target_counts

FPF_TARGET,count_0,count_1
customer_category,,
C01,3923,57
C02,1987,2011
C03,3573,157
C04,3775,218


In [5]:
# Extract features and labels
X = df.drop(columns=["FPF_TARGET", "masked_consumer_id", "customer_category"]).select_dtypes(include=[np.number]).values.astype(np.float32)
y = df["FPF_TARGET"].values.astype(np.float32)

In [6]:
# Stratified train/validation split (80/20)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [7]:
# Logistic regression pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

In [8]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('logreg',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [9]:
# Predict probabilities
y_probs = pipeline.predict_proba(X_val)[:, 1]

In [10]:
# Precision-recall curve
prec, rec, thresholds = precision_recall_curve(y_val, y_probs)
f1_scores = 2 * (prec * rec) / (prec + rec + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"Best threshold: {best_threshold:.4f}")
print(f"Precision at best F1: {prec[best_idx]:.4f}")
print(f"Recall at best F1: {rec[best_idx]:.4f}")
print(f"F1 score: {f1_scores[best_idx]:.4f}")

Best threshold: 0.6523
Precision at best F1: 0.4620
Recall at best F1: 0.6953
F1 score: 0.5551


In [11]:
# Final predictions using tuned threshold
y_pred_custom = (y_probs >= best_threshold).astype(int)

In [12]:
roc_auc_score(y_train, pipeline.decision_function(X_train))

0.8584663940439654

In [13]:
# Evaluate
print("\n=== Validation Report (Custom Threshold) ===")
print(classification_report(y_val, y_pred_custom, digits=4))


=== Validation Report (Custom Threshold) ===
              precision    recall  f1-score   support

         0.0     0.9380    0.8507    0.8922      2652
         1.0     0.4620    0.6953    0.5551       489

    accuracy                         0.8265      3141
   macro avg     0.7000    0.7730    0.7237      3141
weighted avg     0.8639    0.8265    0.8397      3141



In [14]:
roc_auc_score(y_val, pipeline.decision_function(X_val))

0.8489414170576205

In [15]:
# Predict on training data
y_train_probs = pipeline.predict_proba(X_train)[:, 1]
y_train_pred = (y_train_probs >= 0.5).astype(int)

# Evaluate on training set
train_accuracy = accuracy_score(y_train, y_train_pred)
train_report = classification_report(y_train, y_train_pred, output_dict=True)
train_report_df = pd.DataFrame(train_report).transpose()

train_accuracy, train_report_df

(0.7569267515923567,
               precision    recall  f1-score       support
 0.0            0.963431  0.740241  0.837217  10606.000000
 1.0            0.375425  0.847492  0.520346   1954.000000
 accuracy       0.756927  0.756927  0.756927      0.756927
 macro avg      0.669428  0.793867  0.678781  12560.000000
 weighted avg   0.871953  0.756927  0.787920  12560.000000)

### Different Models:

In [16]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import precision_recall_curve, classification_report, roc_auc_score, accuracy_score

# Define customer categories
categories = ["C01", "C02", "C03", "C04"]

# Define models to try
model_pipelines = {
    "LogisticRegression": Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, class_weight='balanced', C=0.1))
    ]),
    "DecisionTree": Pipeline([
        ('scaler', StandardScaler()),
        ('model', DecisionTreeClassifier(class_weight='balanced', random_state=42))
    ]),
    "SVM": Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(probability=True, class_weight='balanced', random_state=42))
    ]),
    "RandomForest": Pipeline([
        ('scaler', StandardScaler()),
        ('model', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))
    ]),
    "GradientBoosting": Pipeline([
        ('scaler', StandardScaler()),
        ('model', GradientBoostingClassifier(n_estimators=200, random_state=42))
    ])
}

# To store results
results = {}

for cat in categories:
    print(f"\n=== Training for Category: {cat} ===")
    
    # Subset the dataframe
    df_cat = df[df["customer_category"] == cat]

    # Skip if not enough samples
    if df_cat.shape[0] < 10:
        print(f"Not enough samples for {cat}, skipping.")
        continue

    # Extract features and labels
    X = df_cat.drop(columns=["masked_consumer_id", "customer_category", "FPF_TARGET"]).select_dtypes(include=[np.number]).values.astype(np.float32)
    y = df_cat["FPF_TARGET"].values.astype(np.float32)

    # Stratified train/validation split
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    results[cat] = {}

    for model_name, pipeline in model_pipelines.items():
        print(f"\n--- Model: {model_name} ---")

        # Train
        pipeline.fit(X_train, y_train)

        # Predict probabilities
        y_probs = pipeline.predict_proba(X_val)[:, 1]

        # Precision-recall curve to find best threshold
        prec, rec, thresholds = precision_recall_curve(y_val, y_probs)
        f1_scores = 2 * (prec * rec) / (prec + rec + 1e-8)
        best_idx = np.argmax(f1_scores)
        best_threshold = thresholds[best_idx]

        # Predict using custom threshold
        y_pred_custom = (y_probs >= best_threshold).astype(int)

        # Evaluation
        val_report = classification_report(y_val, y_pred_custom, digits=4, output_dict=True)

        # Use decision_function if available, otherwise fallback to y_probs
        # Predict on train
        y_train_probs = pipeline.predict_proba(X_train)[:, 1]

        # Compute AUCs
        if hasattr(pipeline.named_steps['model'], 'decision_function'):
            train_auc = roc_auc_score(y_train, pipeline.decision_function(X_train))
            val_auc = roc_auc_score(y_val, pipeline.decision_function(X_val))
        else:
            train_auc = roc_auc_score(y_train, y_train_probs)
            val_auc = roc_auc_score(y_val, y_probs)

        # Store results
        results[cat][model_name] = {
            "best_threshold": best_threshold,
            "precision_at_best": prec[best_idx],
            "recall_at_best": rec[best_idx],
            "f1_at_best": f1_scores[best_idx],
            "train_auc": train_auc,
            "validation_auc": val_auc,
            "validation_report": pd.DataFrame(val_report).transpose()
        }


        # Print key metrics
        print(f"Best threshold: {best_threshold:.4f}")
        print(f"Precision at best F1: {prec[best_idx]:.4f}")
        print(f"Recall at best F1: {rec[best_idx]:.4f}")
        print(f"F1 score: {f1_scores[best_idx]:.4f}")
        print(f"Train AUC: {train_auc:.4f}")
        print(f"Validation AUC: {val_auc:.4f}")


=== Training for Category: C01 ===

--- Model: LogisticRegression ---
Best threshold: 0.8694
Precision at best F1: 0.0294
Recall at best F1: 0.0909
F1 score: 0.0444
Train AUC: 0.9409
Validation AUC: 0.5297

--- Model: DecisionTree ---


/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Best threshold: 0.0000
Precision at best F1: 0.0138
Recall at best F1: 1.0000
F1 score: 0.0273
Train AUC: 1.0000
Validation AUC: 0.4924

--- Model: SVM ---
Best threshold: 0.0267
Precision at best F1: 0.0833
Recall at best F1: 0.0909
F1 score: 0.0870
Train AUC: 0.9991
Validation AUC: 0.6108

--- Model: RandomForest ---
Best threshold: 0.0700
Precision at best F1: 0.0952
Recall at best F1: 0.1818
F1 score: 0.1250
Train AUC: 1.0000
Validation AUC: 0.7124

--- Model: GradientBoosting ---
Best threshold: 0.2379
Precision at best F1: 0.1111
Recall at best F1: 0.0909
F1 score: 0.1000
Train AUC: 1.0000
Validation AUC: 0.6039

=== Training for Category: C02 ===

--- Model: LogisticRegression ---
Best threshold: 0.0686
Precision at best F1: 0.5057
Recall at best F1: 0.9876
F1 score: 0.6689
Train AUC: 0.6898
Validation AUC: 0.5723

--- Model: DecisionTree ---


/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Best threshold: 0.0000
Precision at best F1: 0.5025
Recall at best F1: 1.0000
F1 score: 0.6689
Train AUC: 1.0000
Validation AUC: 0.5199

--- Model: SVM ---
Best threshold: 0.3423
Precision at best F1: 0.5172
Recall at best F1: 0.9726
F1 score: 0.6753
Train AUC: 0.8086
Validation AUC: 0.5767

--- Model: RandomForest ---
Best threshold: 0.2700
Precision at best F1: 0.5096
Recall at best F1: 0.9950
F1 score: 0.6740
Train AUC: 1.0000
Validation AUC: 0.6207

--- Model: GradientBoosting ---
Best threshold: 0.3047
Precision at best F1: 0.5356
Recall at best F1: 0.9179
F1 score: 0.6764
Train AUC: 0.9500
Validation AUC: 0.5951

=== Training for Category: C03 ===

--- Model: LogisticRegression ---
Best threshold: 0.7266
Precision at best F1: 0.1220
Recall at best F1: 0.3226
F1 score: 0.1770
Train AUC: 0.8679
Validation AUC: 0.5876

--- Model: DecisionTree ---


/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Best threshold: 0.0000
Precision at best F1: 0.0416
Recall at best F1: 1.0000
F1 score: 0.0798
Train AUC: 1.0000
Validation AUC: 0.5190

--- Model: SVM ---
Best threshold: 0.0901
Precision at best F1: 0.1268
Recall at best F1: 0.2903
F1 score: 0.1765
Train AUC: 0.9700
Validation AUC: 0.6217

--- Model: RandomForest ---
Best threshold: 0.0600
Precision at best F1: 0.0862
Recall at best F1: 0.4839
F1 score: 0.1463
Train AUC: 1.0000
Validation AUC: 0.6495

--- Model: GradientBoosting ---
Best threshold: 0.0419
Precision at best F1: 0.1261
Recall at best F1: 0.4839
F1 score: 0.2000
Train AUC: 0.9980
Validation AUC: 0.6647

=== Training for Category: C04 ===

--- Model: LogisticRegression ---
Best threshold: 0.6955
Precision at best F1: 0.1134
Recall at best F1: 0.2500
F1 score: 0.1560
Train AUC: 0.8169
Validation AUC: 0.5411

--- Model: DecisionTree ---


/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Best threshold: 0.0000
Precision at best F1: 0.0551
Recall at best F1: 1.0000
F1 score: 0.1044
Train AUC: 1.0000
Validation AUC: 0.4796

--- Model: SVM ---
Best threshold: 0.0584
Precision at best F1: 0.0734
Recall at best F1: 0.5455
F1 score: 0.1294
Train AUC: 0.9775
Validation AUC: 0.5759

--- Model: RandomForest ---
Best threshold: 0.0400
Precision at best F1: 0.0686
Recall at best F1: 0.8182
F1 score: 0.1265
Train AUC: 1.0000
Validation AUC: 0.5839

--- Model: GradientBoosting ---
Best threshold: 0.0374
Precision at best F1: 0.0858
Recall at best F1: 0.4545
F1 score: 0.1444
Train AUC: 0.9997
Validation AUC: 0.6085
